# K-Means reutilizável — pipeline genérico de segmentação

Versão melhorada do notebook original (Mall Customers, 2 features fixas) para
funcionar em **qualquer cenário de clustering tabular**: RFM de clientes,
segmentação de funcionários, agrupamento de sensores, produtos, etc.

**O que mudou em relação ao original:**

- Antes: colunas hardcoded (`"Annual Income (k$)"`, `"Spending Score (1-100)"`) espalhadas pelo código. Agora: um único objeto `ClusterConfig` no topo — trocar de cenário é trocar esse objeto, nenhuma célula abaixo muda.
- Antes: só 2 features numéricas, sem suporte a categórica. Agora: `ColumnTransformer` genérico — N features numéricas + N categóricas (one-hot), qualquer quantidade.
- Antes: sem tratamento de outlier. Agora: outlier é uma decisão explícita e documentada (`outlier_method`), aplicada antes do scaling.
- Antes: escolha de k só por `argmax(silhouette)`, sem checar se o resultado é estável. Agora: mesma escolha, mas com um teste de **estabilidade** (Adjusted Rand Index entre reruns) — silhouette alto com estabilidade baixa é sinal de alerta, não de sucesso.
- Antes: visualização só funcionava com 2 features. Agora: com 3+ features, projeta em PCA 2D só para visualizar (o modelo continua treinado no espaço original — PCA é lente, não input do KMeans).
- Antes: modelo morria no notebook. Agora: `save_pipeline` / `load_pipeline` / `predict_new` — o cluster de um cliente novo é calculado sem re-treinar nada.
- Antes: a animação em GIF (reimplementação manual do algoritmo) misturada com o pipeline "de verdade". Agora: isolada numa seção **opcional** no fim, claramente separada — é material didático, não faz parte do que roda em produção.

**Workflow seguido** (Definir → Perfilar → Tratar → Engenheirar → Selecionar/Treinar → Avaliar → Comunicar → Persistir), célula por seção abaixo.


In [ ]:
from __future__ import annotations

import pickle
import warnings
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Definir o problema — `ClusterConfig`

Tudo que muda entre cenários mora aqui. `numeric_features` e `categorical_features`
aceitam qualquer quantidade de colunas (inclusive zero categóricas, como no
notebook original).

In [ ]:
@dataclass
class ClusterConfig:
    '''Configuração de um cenário de clustering. Trocar cenário = trocar esta
    instância; nenhuma função abaixo precisa ser editada.'''
    data_path: str
    id_col: Optional[str] = None
    numeric_features: list = field(default_factory=list)
    categorical_features: list = field(default_factory=list)
    outlier_method: str = "iqr"        # "iqr" | "zscore" | "none"
    outlier_factor: float = 1.5
    scale_method: str = "standard"     # "standard" | "robust" (robust se houver outlier pesado)
    k_range: range = range(2, 9)
    random_state: int = 42
    n_init: int = 10
    model_path: str = "cluster_pipeline.pkl"


# CONFIG — reproduz o notebook original (Renda x Spending Score)
# CONFIG_CELL_MARKER
CONFIG = ClusterConfig(
    data_path="dataset/Mall_Customers.csv",
    id_col="CustomerID",
    numeric_features=["Annual Income (k$)", "Spending Score (1-100)"],
    categorical_features=[],
)

## 2. Perfilar os dados

Antes de tratar qualquer coisa, checa tipo, % de nulo e cardinalidade das
colunas que vão entrar no modelo. K-Means não lida com `NaN` — se aparecer
nulo aqui, decida (imputar ou remover) antes de seguir.

In [ ]:
def load_data(config: ClusterConfig) -> pd.DataFrame:
    df = pd.read_csv(config.data_path)
    used_cols = config.numeric_features + config.categorical_features
    missing = [c for c in used_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Colunas ausentes no dataset: {missing}")
    return df


def profile_dataset(df: pd.DataFrame, config: ClusterConfig) -> pd.DataFrame:
    cols = config.numeric_features + config.categorical_features
    prof = pd.DataFrame({
        "dtype": df[cols].dtypes.astype(str),
        "pct_nulo": (df[cols].isna().mean() * 100).round(2),
        "n_unicos": df[cols].nunique(),
    })
    if prof["pct_nulo"].gt(0).any():
        warnings.warn("Há colunas com nulo nas features do modelo — trate antes de escalar.")
    return prof

## 3. Tratar outliers — decisão de negócio, não só estatística

K-Means usa a **média** como centróide: um outlier puxa o cluster inteiro na
direção dele. `handle_outliers` recorta (clip) os valores fora do intervalo
antes do scaling. O ponto importante: decidir *por que* um valor é outlier —
erro de digitação (recortar/remover é certo) ou cliente legítimo raro (nesse
caso considere isolar como segmento à parte em vez de simplesmente cortar).

In [ ]:
def handle_outliers(df: pd.DataFrame, config: ClusterConfig):
    if config.outlier_method == "none" or not config.numeric_features:
        return df.copy(), pd.DataFrame()

    df = df.copy()
    report = {}
    for col in config.numeric_features:
        if config.outlier_method == "iqr":
            q1, q3 = df[col].quantile([0.25, 0.75])
            iqr = q3 - q1
            low, high = q1 - config.outlier_factor * iqr, q3 + config.outlier_factor * iqr
        elif config.outlier_method == "zscore":
            mean, std = df[col].mean(), df[col].std()
            low, high = mean - config.outlier_factor * std, mean + config.outlier_factor * std
        else:
            raise ValueError(f"outlier_method invalido: {config.outlier_method}")

        n_affected = int(((df[col] < low) | (df[col] > high)).sum())
        report[col] = {"limite_inf": round(float(low), 2), "limite_sup": round(float(high), 2),
                        "n_pontos_ajustados": n_affected}
        df[col] = df[col].clip(low, high)

    return df, pd.DataFrame(report).T

## 4. Engenharia de features — scaling + encoding genéricos

`ColumnTransformer` é o que generaliza o notebook: escala as numéricas
(`StandardScaler` ou `RobustScaler`) e faz one-hot das categóricas, para
qualquer combinação de colunas — 1, 2 ou 30, numéricas e/ou categóricas.

In [ ]:
def build_preprocessor(config: ClusterConfig) -> ColumnTransformer:
    scaler = StandardScaler() if config.scale_method == "standard" else RobustScaler()

    transformers = []
    if config.numeric_features:
        transformers.append(("num", scaler, config.numeric_features))
    if config.categorical_features:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                              config.categorical_features))

    if not transformers:
        raise ValueError("Configure ao menos uma feature numerica ou categorica.")

    return ColumnTransformer(transformers)

## 5. Selecionar k — elbow + silhouette

Mesma lógica do original (inércia + silhouette por k), agora reutilizável
para qualquer `X`. `suggest_k` é ponto de partida, não veredito final — releia
o gráfico antes de aceitar.

In [ ]:
def evaluate_k(X: np.ndarray, config: ClusterConfig, plot: bool = True) -> pd.DataFrame:
    rows = []
    for k in config.k_range:
        km = KMeans(n_clusters=k, n_init=config.n_init, random_state=config.random_state)
        labels = km.fit_predict(X)
        rows.append({"k": k, "inertia": km.inertia_, "silhouette": silhouette_score(X, labels)})
    result = pd.DataFrame(rows)

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].plot(result["k"], result["inertia"], marker="o")
        axes[0].set(title="Elbow (inercia)", xlabel="k", ylabel="Inercia")
        axes[1].plot(result["k"], result["silhouette"], marker="o", color="darkorange")
        axes[1].set(title="Silhouette por k", xlabel="k", ylabel="Silhouette")
        plt.tight_layout()
        plt.show()

    return result


def suggest_k(k_eval: pd.DataFrame) -> int:
    return int(k_eval.loc[k_eval["silhouette"].idxmax(), "k"])

## 6. Treinar o modelo final

In [ ]:
def fit_kmeans(X: np.ndarray, k: int, config: ClusterConfig) -> KMeans:
    km = KMeans(n_clusters=k, n_init=config.n_init, random_state=config.random_state)
    km.fit(X)
    return km

## 7. Avaliar — silhouette final + estabilidade

Silhouette mede separação. **Estabilidade** mede outra coisa: se você treinar
de novo com sementes diferentes, os clusters continuam os mesmos? Roda o
KMeans `n_runs` vezes e mede a concordância entre as partições via
Adjusted Rand Index (1 = idênticas, ~0 = ao acaso). Silhouette ok + estabilidade
baixa é sinal de que os clusters não são robustos — normalmente indica k mal
escolhido ou dados sem estrutura de grupo clara.

In [ ]:
def evaluate_stability(X: np.ndarray, k: int, config: ClusterConfig, n_runs: int = 5) -> float:
    runs = []
    for seed in range(n_runs):
        km = KMeans(n_clusters=k, n_init=config.n_init, random_state=seed)
        runs.append(km.fit_predict(X))

    scores = [adjusted_rand_score(runs[i], runs[j])
              for i in range(n_runs) for j in range(i + 1, n_runs)]
    return float(np.mean(scores))

## 8. Visualizar

Com 2 features, plota os eixos originais (como no notebook original). Com 3+,
projeta em PCA 2D **só para visualização** — o modelo continua treinado no
espaço original de N dimensões; PCA aqui é lente, não vira input do KMeans.

In [ ]:
def plot_clusters(X: np.ndarray, labels: np.ndarray, config: ClusterConfig) -> None:
    if X.shape[1] == 2:
        coords, xlabel, ylabel = X, "feature 1 (padronizada)", "feature 2 (padronizada)"
    else:
        pca = PCA(n_components=2, random_state=config.random_state)
        coords = pca.fit_transform(X)
        var = pca.explained_variance_ratio_.sum() * 100
        xlabel, ylabel = "PC1", f"PC2  ({var:.1f}% da variancia em 2 componentes)"

    plt.figure(figsize=(6, 5))
    for c in sorted(np.unique(labels)):
        m = labels == c
        plt.scatter(coords[m, 0], coords[m, 1], s=40, label=f"Cluster {c}")
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.title(f"Clusters (k={len(np.unique(labels))})")
    plt.legend(fontsize=8)
    plt.show()

## 9. Comunicar — perfil de negócio por cluster

O output que interessa fora do notebook: tamanho, % da base, médias das
numéricas e categoria dominante das categóricas, por cluster.

In [ ]:
def profile_clusters(df: pd.DataFrame, labels: np.ndarray, config: ClusterConfig) -> pd.DataFrame:
    work = df.copy()
    work["cluster"] = labels

    profile = work.groupby("cluster").agg(n_registros=("cluster", "size"))
    profile["pct_base"] = (profile["n_registros"] / len(work) * 100).round(1)

    for col in config.numeric_features:
        profile[f"{col}_media"] = work.groupby("cluster")[col].mean().round(2)

    for col in config.categorical_features:
        profile[f"{col}_dominante"] = work.groupby("cluster")[col].agg(
            lambda s: s.mode().iat[0] if not s.mode().empty else None
        )

    return profile.sort_index()

## 10. Persistir e reaplicar

`save_pipeline` grava o preprocessor + modelo treinados. `predict_new` aplica
os dois a um registro novo **sem re-treinar e sem re-escolher k** — é isso que
torna o notebook reutilizável em produção, não só numa análise pontual.

In [ ]:
def save_pipeline(preprocessor: ColumnTransformer, model: KMeans, config: ClusterConfig) -> None:
    with open(config.model_path, "wb") as f:
        pickle.dump({"preprocessor": preprocessor, "model": model, "config": config}, f)


def load_pipeline(model_path: str) -> dict:
    with open(model_path, "rb") as f:
        return pickle.load(f)


def predict_new(df_new: pd.DataFrame, bundle: dict) -> np.ndarray:
    config = bundle["config"]
    cols = config.numeric_features + config.categorical_features
    X_new = bundle["preprocessor"].transform(df_new[cols])
    return bundle["model"].predict(X_new)

## Orquestrador — roda o workflow inteiro

Junta as seções 2-10 numa chamada. `k=None` deixa `suggest_k` decidir; passe
um inteiro para fixar k manualmente (ex.: quando o negócio já sabe quantos
segmentos quer).

In [ ]:
def run_clustering_pipeline(config: ClusterConfig, k: Optional[int] = None,
                             show_plots: bool = True) -> dict:
    df = load_data(config)
    profile_dataset(df, config)

    df_clean, outlier_report = handle_outliers(df, config)

    preprocessor = build_preprocessor(config)
    cols = config.numeric_features + config.categorical_features
    X = preprocessor.fit_transform(df_clean[cols])

    k_eval = evaluate_k(X, config, plot=show_plots)
    k_final = k or suggest_k(k_eval)

    model = fit_kmeans(X, k_final, config)
    labels = model.labels_

    stability = evaluate_stability(X, k_final, config)
    sil_final = silhouette_score(X, labels)

    if show_plots:
        plot_clusters(X, labels, config)

    cluster_profile = profile_clusters(df_clean, labels, config)
    save_pipeline(preprocessor, model, config)

    return {
        "df_labeled": df_clean.assign(cluster=labels),
        "k_eval": k_eval,
        "k_final": k_final,
        "silhouette_final": sil_final,
        "stability_ari": stability,
        "outlier_report": outlier_report,
        "cluster_profile": cluster_profile,
        "model": model,
        "preprocessor": preprocessor,
    }

## Exemplo 1 — cenário original (retrocompatibilidade)

Mesmo resultado do notebook antigo: 2 features, sem categórica.

In [ ]:
result = run_clustering_pipeline(CONFIG)
print(f"k escolhido: {result['k_final']}  |  silhouette: {result['silhouette_final']:.3f}"
      f"  |  estabilidade (ARI): {result['stability_ari']:.3f}")
result["cluster_profile"]

## Exemplo 2 — outro cenário, só trocando CONFIG

Mesma base, mas agora com 3 numéricas + 1 categórica (`Genre`) e
`RobustScaler` (mais resistente a outlier). Nenhuma função acima foi
tocada — só o `ClusterConfig`.

Para aplicar em RFM (Recência/Frequência/Valor), segmentação de funcionários,
sensores, etc.: troque `data_path`, `numeric_features` e `categorical_features`
por suas colunas. Se o cenário for especificamente RFM de clientes, a skill
`analise-rfm` já cobre esse caso com tratamento de outlier como segmento de
negócio e perfil de % de receita por segmento.

In [ ]:
CONFIG_V2 = ClusterConfig(
    data_path="dataset/Mall_Customers.csv",
    id_col="CustomerID",
    numeric_features=["Age", "Annual Income (k$)", "Spending Score (1-100)"],
    categorical_features=["Genre"],
    outlier_method="iqr",
    scale_method="robust",
    model_path="cluster_pipeline_v2.pkl",
)

result_v2 = run_clustering_pipeline(CONFIG_V2)
result_v2["cluster_profile"]

## Aplicando o modelo salvo a um registro novo, sem re-treinar

O dicionário abaixo é um template: as chaves precisam bater com as colunas do
`CONFIG` que você usou para treinar (`numeric_features` + `categorical_features`).
Aqui está com as 2 colunas do `CONFIG` original — se você trocou o `CONFIG` lá em
cima para outro cenário, atualize as chaves para as colunas dele.

In [ ]:
bundle = load_pipeline(CONFIG.model_path)
novo_cliente = pd.DataFrame([{"Annual Income (k$)": 80, "Spending Score (1-100)": 75}])
cluster_previsto = predict_new(novo_cliente, bundle)
print("Cluster do novo cliente:", cluster_previsto[0])

## Limitações — vale saber antes de aplicar em qualquer dado

K-Means minimiza variância intra-cluster, o que assume implicitamente
clusters **esféricos e de tamanho/densidade parecidos**. Não lida bem com
formatos alongados, aninhados ou densidades muito desiguais — nesses casos
considere `DBSCAN` (densidade, sem k fixo, mas sensível a `eps`) ou
`GaussianMixture` (clusters elípticos, atribuição probabilística).

É sensível a outlier (centróide é média — trate na seção 3) e à
dimensionalidade alta (acima de ~10-15 features correlacionadas, distância
euclidiana perde poder discriminante — considere PCA antes do clustering,
não só para visualizar). k precisa ser escolhido a priori; use a seção 5 +
estabilidade (seção 7) como apoio, não como decisão automática.

## Extra didático (opcional) — animação do algoritmo de Lloyd

Isolado do pipeline principal de propósito: não tem função analítica, é só
visual/educacional (mostra os centróides convergindo iteração a iteração).
Requer `imageio` (`pip install imageio`). Não é chamado por
`run_clustering_pipeline` nem é necessário para reusar o pipeline em outro
cenário.

In [ ]:
def animate_kmeans_lloyd(X: np.ndarray, k: int, max_iters: int = 15, seed: int = 42,
                          gif_path: str = "kmeans_animation.gif", folder: str = "kmeans_steps"):
    import os
    import imageio
    from IPython.display import Image

    def kmeans_pp_init(X, k, rng):
        n = X.shape[0]
        centroids = np.empty((k, X.shape[1]))
        centroids[0] = X[rng.randint(n)]
        for i in range(1, k):
            d2 = np.min(np.linalg.norm(X[:, None] - centroids[:i], axis=2) ** 2, axis=1)
            probs = d2 / d2.sum()
            idx = rng.choice(n, p=probs)
            centroids[i] = X[idx]
        return centroids

    os.makedirs(folder, exist_ok=True)
    rng = np.random.RandomState(seed)
    centroids = kmeans_pp_init(X, k, rng)
    snapshots = []

    for i in range(1, max_iters + 1):
        dists = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)
        labels = np.argmin(dists, axis=1)

        plt.figure(figsize=(6, 5))
        for c in range(k):
            plt.scatter(X[labels == c, 0], X[labels == c, 1], s=40, label=f"Cluster {c}")
        plt.scatter(centroids[:, 0], centroids[:, 1], c="black", marker="*", s=200, label="Centroides")
        plt.title(f"K-Means iteracao {i}")
        plt.legend(loc="best", fontsize=8)

        fname = f"{folder}/iter_{i:02d}.png"
        plt.savefig(fname, dpi=140, bbox_inches="tight")
        plt.close()
        snapshots.append(fname)

        new_centroids = np.vstack([
            X[labels == c].mean(axis=0) if np.any(labels == c) else centroids[c]
            for c in range(k)
        ])
        if np.allclose(centroids, new_centroids, atol=1e-4):
            break
        centroids = new_centroids

    with imageio.get_writer(gif_path, mode="I", duration=2000, loop=0) as writer:
        for fname in snapshots:
            writer.append_data(imageio.v2.imread(fname))

    return Image(filename=gif_path)

# animate_kmeans_lloyd(X, k=result['k_final'])  # descomente para gerar o GIF